# Kaggle Training: Layer-Aware FD-Loss MNIST

Upload this notebook alone to Kaggle and run all cells. It bootstraps the small local project into `/kaggle/working/fd-loss-mnist`, trains the classifier/statistics/generator pipeline, evaluates the final generator with fresh samples, and writes one downloadable artifact:

```text
/kaggle/working/fd_loss_mnist_run.zip
```

In [1]:
from pathlib import Path
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from datetime import datetime

import numpy as np
import torch

IS_KAGGLE = Path("/kaggle/working").exists()
PROJECT_DIR = Path("/kaggle/working/fd-loss-mnist") if IS_KAGGLE else Path.cwd().resolve().parent
RUNS_DIR = Path("/kaggle/working/runs") if IS_KAGGLE else PROJECT_DIR / "runs"
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else PROJECT_DIR
RUN_NAME = os.environ.get("FD_RUN_NAME", "layer_fd_kaggle")
SEED = int(os.environ.get("FD_SEED", "1"))
FAST_DEV_RUN = os.environ.get("FD_FAST_DEV_RUN", "0") == "1"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Kaggle:", IS_KAGGLE)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)
print("Working directory:", Path.cwd())
print("Project directory:", PROJECT_DIR)
print("Runs directory:", RUNS_DIR)

Kaggle: True
Python: 3.12.12
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
Device: cuda
Working directory: /kaggle/working
Project directory: /kaggle/working/fd-loss-mnist
Runs directory: /kaggle/working/runs


In [2]:
from pathlib import Path

BOOTSTRAP_FILES = {
  "src/__init__.py": "\"\"\"Reusable code for layer-aware FD-loss experiments.\"\"\"\n",
  "src/data.py": "from __future__ import annotations\n\nimport gzip\nimport struct\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom torchvision import datasets, transforms\n\nfrom .utils import PROJECT_ROOT, resolve_path\n\n\n@dataclass\nclass DataBundle:\n    \"\"\"Train/test datasets, loaders, and tensor views for quick analysis.\"\"\"\n\n    train_dataset: torch.utils.data.Dataset\n    test_dataset: torch.utils.data.Dataset\n    train_loader: DataLoader\n    test_loader: DataLoader\n    x_train: torch.Tensor\n    y_train: torch.Tensor\n    x_test: torch.Tensor\n    y_test: torch.Tensor\n    name: str\n    num_classes: int\n\n\ndef resolve_idx_file(path: Path) -> Path:\n    \"\"\"Find an IDX file, including Kaggle's occasional file-as-folder layout.\"\"\"\n    path = Path(path)\n    if path.is_file():\n        return path\n    if path.is_dir():\n        files = [p for p in path.rglob(\"*\") if p.is_file()]\n        if len(files) == 1:\n            return files[0]\n        for p in files:\n            if path.name in p.name:\n                return p\n        if files:\n            return files[0]\n    raise FileNotFoundError(f\"Could not resolve IDX file from {path}\")\n\n\ndef read_idx_images(path: str | Path) -> torch.Tensor:\n    \"\"\"Read MNIST IDX image files into [N, 1, H, W] float tensors.\"\"\"\n    path = resolve_idx_file(Path(path))\n    opener = gzip.open if path.suffix == \".gz\" else open\n    with opener(path, \"rb\") as f:\n        magic, n, rows, cols = struct.unpack(\">IIII\", f.read(16))\n        if magic != 2051:\n            raise ValueError(f\"Bad image magic number: {magic}\")\n        data = np.frombuffer(f.read(), dtype=np.uint8)\n    return torch.tensor(data.copy(), dtype=torch.float32).view(n, 1, rows, cols) / 255.0\n\n\ndef read_idx_labels(path: str | Path) -> torch.Tensor:\n    \"\"\"Read MNIST IDX label files into long tensors.\"\"\"\n    path = resolve_idx_file(Path(path))\n    opener = gzip.open if path.suffix == \".gz\" else open\n    with opener(path, \"rb\") as f:\n        magic, _ = struct.unpack(\">II\", f.read(8))\n        if magic != 2049:\n            raise ValueError(f\"Bad label magic number: {magic}\")\n        data = np.frombuffer(f.read(), dtype=np.uint8)\n    return torch.tensor(data.copy(), dtype=torch.long)\n\n\ndef _idx_paths_exist(idx_path: Path) -> bool:\n    expected = [\n        \"train-images-idx3-ubyte\",\n        \"train-labels-idx1-ubyte\",\n        \"t10k-images-idx3-ubyte\",\n        \"t10k-labels-idx1-ubyte\",\n    ]\n    return idx_path.exists() and all((idx_path / name).exists() for name in expected)\n\n\ndef discover_mnist_idx_dir(preferred: str | Path | None = None) -> Path | None:\n    \"\"\"Find a directory containing MNIST IDX files in common Kaggle layouts.\"\"\"\n    names = [\n        \"train-images-idx3-ubyte\",\n        \"train-images-idx3-ubyte.gz\",\n        \"train-labels-idx1-ubyte\",\n        \"train-labels-idx1-ubyte.gz\",\n        \"t10k-images-idx3-ubyte\",\n        \"t10k-images-idx3-ubyte.gz\",\n        \"t10k-labels-idx1-ubyte\",\n        \"t10k-labels-idx1-ubyte.gz\",\n    ]\n    roots = []\n    if preferred:\n        roots.append(Path(preferred))\n    roots.extend([Path(\"/kaggle/input\"), Path(\"data\")])\n\n    for root in roots:\n        if not root.exists():\n            continue\n        candidates = [root] if root.is_dir() else []\n        candidates.extend([p for p in root.rglob(\"*\") if p.is_dir()])\n        for candidate in candidates:\n            found = {p.name for p in candidate.iterdir() if p.is_file() or p.is_dir()}\n            has_train_images = any(name in found for name in names[:2])\n            has_train_labels = any(name in found for name in names[2:4])\n            has_test_images = any(name in found for name in names[4:6])\n            has_test_labels = any(name in found for name in names[6:8])\n            if has_train_images and has_train_labels and has_test_images and has_test_labels:\n                return candidate\n    return None\n\n\ndef _first_existing_idx(base: Path, names: list[str]) -> Path:\n    for name in names:\n        path = base / name\n        if path.exists():\n            return path\n    raise FileNotFoundError(f\"Could not find any of {names} under {base}\")\n\n\ndef load_mnist(config: dict, root: str | Path = PROJECT_ROOT) -> DataBundle:\n    \"\"\"Load MNIST from Kaggle IDX files when present, otherwise torchvision.\"\"\"\n    dataset_cfg = config.get(\"dataset\", config)\n    num_classes = int(config.get(\"num_classes\", dataset_cfg.get(\"num_classes\", 10)))\n    idx_path = Path(dataset_cfg.get(\"idx_path\", \"/kaggle/input/datasets/hojjatk/mnist-dataset\"))\n    tv_dir = resolve_path(dataset_cfg.get(\"torchvision_dir\", \"data\"), root)\n    train_bs = int(dataset_cfg.get(\"train_batch_size\", 256))\n    test_bs = int(dataset_cfg.get(\"test_batch_size\", 512))\n    num_workers = int(dataset_cfg.get(\"num_workers\", 0))\n\n    discovered_idx_path = discover_mnist_idx_dir(idx_path)\n\n    if discovered_idx_path is not None:\n        print(f\"Loading MNIST IDX files from {discovered_idx_path}\")\n        x_train = read_idx_images(_first_existing_idx(discovered_idx_path, [\"train-images-idx3-ubyte\", \"train-images-idx3-ubyte.gz\"]))\n        y_train = read_idx_labels(_first_existing_idx(discovered_idx_path, [\"train-labels-idx1-ubyte\", \"train-labels-idx1-ubyte.gz\"]))\n        x_test = read_idx_images(_first_existing_idx(discovered_idx_path, [\"t10k-images-idx3-ubyte\", \"t10k-images-idx3-ubyte.gz\"]))\n        y_test = read_idx_labels(_first_existing_idx(discovered_idx_path, [\"t10k-labels-idx1-ubyte\", \"t10k-labels-idx1-ubyte.gz\"]))\n        train_dataset = TensorDataset(x_train, y_train)\n        test_dataset = TensorDataset(x_test, y_test)\n    else:\n        print(f\"Loading MNIST through torchvision at {tv_dir}\")\n        transform = transforms.ToTensor()\n        download = bool(dataset_cfg.get(\"download\", True))\n        try:\n            train_dataset = datasets.MNIST(root=tv_dir, train=True, transform=transform, download=download)\n            test_dataset = datasets.MNIST(root=tv_dir, train=False, transform=transform, download=download)\n        except Exception as exc:\n            raise RuntimeError(\n                \"Could not find MNIST IDX files under /kaggle/input and torchvision MNIST \"\n                \"loading failed. Attach a Kaggle MNIST dataset or enable internet/download.\"\n            ) from exc\n        x_train = train_dataset.data.unsqueeze(1).float() / 255.0\n        y_train = train_dataset.targets.long()\n        x_test = test_dataset.data.unsqueeze(1).float() / 255.0\n        y_test = test_dataset.targets.long()\n\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=train_bs,\n        shuffle=True,\n        num_workers=num_workers,\n    )\n    test_loader = DataLoader(\n        test_dataset,\n        batch_size=test_bs,\n        shuffle=False,\n        num_workers=num_workers,\n    )\n    return DataBundle(\n        train_dataset=train_dataset,\n        test_dataset=test_dataset,\n        train_loader=train_loader,\n        test_loader=test_loader,\n        x_train=x_train,\n        y_train=y_train,\n        x_test=x_test,\n        y_test=y_test,\n        name=\"mnist\",\n        num_classes=num_classes,\n    )\n\n\ndef load_data(config: dict, root: str | Path = PROJECT_ROOT) -> DataBundle:\n    \"\"\"Dataset factory.\"\"\"\n    name = config.get(\"dataset\", {}).get(\"name\", \"mnist\")\n    if name.lower() != \"mnist\":\n        raise ValueError(f\"Unsupported dataset: {name}\")\n    return load_mnist(config, root=root)\n",
  "src/eval.py": "from __future__ import annotations\n\nimport csv\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib.pyplot as plt\nimport torch\n\nfrom .fd_loss import frechet_to_real\nfrom .models import build_classifier, build_generator\nfrom .features import build_feature_extractor\nfrom .plotting import (\n    plot_eigenspectrum,\n    plot_real_generated_correlation_triplet,\n    plot_real_generated_covariance_triplet,\n    plot_sample_grid_with_predictions,\n)\nfrom .stats import compute_mean_cov_from_sums, load_stats, sqrtm_psd_nograd\nfrom .utils import PROJECT_ROOT, load_config, resolve_path, save_json\n\n\ndef _load_state_dict(model, checkpoint_path: Path, device: torch.device) -> None:\n    obj = torch.load(checkpoint_path, map_location=device)\n    state = obj.get(\"model_state_dict\", obj) if isinstance(obj, dict) else obj\n    model.load_state_dict(state)\n\n\ndef _write_csv(rows: list[dict[str, Any]], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        return\n    fields = sorted({k for row in rows for k in row})\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef _plot_confusion(confusion: torch.Tensor, path: Path) -> None:\n    fig, ax = plt.subplots(figsize=(6, 5))\n    im = ax.imshow(confusion.cpu().numpy(), cmap=\"Blues\")\n    ax.set_title(\"Classifier confusion on generated samples\")\n    ax.set_xlabel(\"predicted\")\n    ax.set_ylabel(\"conditioning label\")\n    ax.set_xticks(range(confusion.shape[1]))\n    ax.set_yticks(range(confusion.shape[0]))\n    fig.colorbar(im, ax=ax, shrink=0.8)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=170, bbox_inches=\"tight\")\n    plt.close(fig)\n\n\n@torch.no_grad()\ndef evaluate_generator_fresh(\n    generator,\n    classifier,\n    feature_model,\n    real_stats_obj: dict,\n    config: dict,\n    eval_config: dict,\n    output_dir: str | Path,\n    device: torch.device,\n) -> dict[str, Any]:\n    \"\"\"Generate fresh samples and write evaluation artifacts.\"\"\"\n    output_dir = Path(output_dir)\n    (output_dir / \"samples\").mkdir(parents=True, exist_ok=True)\n    plots_dir = output_dir.parent / \"plots\" if output_dir.name == \"eval\" else output_dir / \"plots\"\n    plots_dir.mkdir(parents=True, exist_ok=True)\n\n    real_stats = real_stats_obj[\"stats\"]\n    metadata = real_stats_obj.get(\"metadata\", {})\n    layer_names = list(metadata.get(\"layer_names\", real_stats.keys()))\n    layer_dims = {layer: int(real_stats[layer][\"mu\"].shape[1]) for layer in layer_names}\n    num_classes = int(config.get(\"num_classes\", eval_config.get(\"num_classes\", 10)))\n    sample_cfg = eval_config.get(\"samples\", {})\n    per_class = int(sample_cfg.get(\"per_class\", 256))\n    batch_per_class = int(sample_cfg.get(\"batch_per_class\", 64))\n    z_dim = int(sample_cfg.get(\"z_dim\", config.get(\"generator\", {}).get(\"z_dim\", 64)))\n\n    sums = {layer: torch.zeros(num_classes, layer_dims[layer]) for layer in layer_names}\n    outers = {layer: torch.zeros(num_classes, layer_dims[layer], layer_dims[layer]) for layer in layer_names}\n    counts = {layer: torch.zeros(num_classes, dtype=torch.long) for layer in layer_names}\n    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)\n\n    sample_images = []\n    sample_labels = []\n    sample_preds = []\n    max_grid_per_class = 10\n\n    generator.eval()\n    classifier.eval()\n    feature_model.eval()\n\n    for cls in range(num_classes):\n        remaining = per_class\n        while remaining > 0:\n            n = min(batch_per_class, remaining)\n            y = torch.full((n,), cls, device=device, dtype=torch.long)\n            z = torch.randn(n, z_dim, device=device)\n            x = generator(z, y)\n            logits = classifier(x)\n            preds = logits.argmax(dim=1)\n            features = feature_model(x)\n\n            for pred in preds.cpu():\n                confusion[cls, int(pred)] += 1\n\n            if len([v for v in sample_labels if v == cls]) < max_grid_per_class:\n                take = min(max_grid_per_class - len([v for v in sample_labels if v == cls]), n)\n                sample_images.append(x[:take].detach().cpu())\n                sample_labels.extend([cls] * take)\n                sample_preds.extend(preds[:take].detach().cpu().tolist())\n\n            for layer in layer_names:\n                f = features[layer].detach().cpu().float()\n                sums[layer][cls] += f.sum(dim=0)\n                outers[layer][cls] += f.T @ f\n                counts[layer][cls] += f.shape[0]\n            remaining -= n\n\n    eps = float(config.get(\"train\", {}).get(\"covariance_epsilon\", metadata.get(\"covariance_epsilon\", 1e-4)))\n    gen_stats = {}\n    for layer in layer_names:\n        mu, cov = compute_mean_cov_from_sums(counts[layer], sums[layer], outers[layer], eps=eps)\n        cov_sqrt = torch.stack([sqrtm_psd_nograd(cov[cls]) for cls in range(num_classes)])\n        gen_stats[layer] = {\"mu\": mu, \"cov\": cov, \"cov_sqrt\": cov_sqrt, \"n\": counts[layer], \"dim\": torch.tensor(layer_dims[layer])}\n\n    fd_rows = []\n    fd_total = 0.0\n    layer_weights = {spec[\"name\"]: float(spec.get(\"weight\", 1.0)) for spec in config.get(\"features\", {}).get(\"selected_layers\", [])}\n    for layer in layer_names:\n        layer_total = 0.0\n        for cls in range(num_classes):\n            terms = frechet_to_real(\n                gen_stats[layer][\"mu\"][cls].to(device),\n                gen_stats[layer][\"cov\"][cls].to(device),\n                real_stats[layer],\n                cls,\n            )\n            row = {\n                \"layer\": layer,\n                \"class\": cls,\n                \"n\": int(gen_stats[layer][\"n\"][cls].item()),\n                \"dim\": layer_dims[layer],\n                \"mean_term\": float(terms[\"mean\"].detach().cpu()),\n                \"cov_term\": float(terms[\"cov\"].detach().cpu()),\n                \"total_fd\": float(terms[\"total\"].detach().cpu()),\n            }\n            fd_rows.append(row)\n            layer_total += row[\"total_fd\"]\n        layer_avg = layer_total / num_classes\n        fd_total += layer_weights.get(layer, 1.0) * layer_avg\n\n    _write_csv(fd_rows, output_dir / \"fd_by_layer_class.csv\")\n    _plot_confusion(confusion, plots_dir / \"generated_confusion_matrix.png\")\n\n    if sample_images:\n        images = torch.cat(sample_images, dim=0)\n        labels = torch.tensor(sample_labels)\n        preds = torch.tensor(sample_preds)\n        plot_sample_grid_with_predictions(\n            images,\n            labels,\n            preds,\n            n_per_class=max_grid_per_class,\n            path=output_dir / \"samples\" / \"generated_grid_with_predictions.png\",\n        )\n\n    plot_cfg = eval_config.get(\"plots\", {})\n    layers_to_plot = [layer for layer in plot_cfg.get(\"layers\", layer_names) if layer in layer_names]\n    digits_to_plot = [int(d) for d in plot_cfg.get(\"digits\", [0]) if int(d) < num_classes]\n    percentile = float(plot_cfg.get(\"matrix_percentile\", 99.0))\n\n    for layer in layers_to_plot:\n        for cls in digits_to_plot:\n            real_cov = real_stats[layer][\"cov\"][cls].detach().cpu()\n            gen_cov = gen_stats[layer][\"cov\"][cls].detach().cpu()\n            stem = f\"{layer}_class_{cls}\"\n            plot_real_generated_covariance_triplet(\n                real_cov,\n                gen_cov,\n                title=f\"Covariance {stem}\",\n                path=plots_dir / f\"cov_triplet_{stem}.png\",\n                percentile=percentile,\n            )\n            plot_real_generated_correlation_triplet(\n                real_cov,\n                gen_cov,\n                title=f\"Correlation {stem}\",\n                path=plots_dir / f\"corr_triplet_{stem}.png\",\n                percentile=percentile,\n            )\n            plot_eigenspectrum(\n                {\"real\": real_cov, \"generated\": gen_cov},\n                title=f\"Eigenspectrum {stem}\",\n                path=plots_dir / f\"eigenspectrum_{stem}.png\",\n            )\n\n    summary = {\n        \"fd_total_weighted\": fd_total,\n        \"num_classes\": num_classes,\n        \"per_class\": per_class,\n        \"layer_names\": layer_names,\n        \"confusion_matrix\": confusion.tolist(),\n        \"classifier_fake_acc\": float(confusion.diag().sum().item() / confusion.sum().clamp_min(1).item()),\n    }\n    save_json(summary, output_dir / \"eval_summary.json\")\n    torch.save({\"stats\": gen_stats, \"metadata\": {\"source\": \"generated_fresh\", \"per_class\": per_class}}, output_dir / \"generated_stats.pt\")\n    return summary\n\n\ndef run_evaluation(run_dir: str | Path, eval_config_path: str | Path | None = None, root: str | Path = PROJECT_ROOT) -> dict[str, Any]:\n    \"\"\"Load a run folder and evaluate its generator independent of training state.\"\"\"\n    run_dir = Path(run_dir)\n    config = load_config(run_dir / \"config.yaml\")\n    eval_config = load_config(eval_config_path) if eval_config_path else load_config(Path(root) / \"configs\" / \"eval_mnist.yaml\")\n    device = torch.device(eval_config.get(\"device\", config.get(\"device\", \"cpu\")) if eval_config.get(\"device\", \"auto\") != \"auto\" else (\"cuda\" if torch.cuda.is_available() else \"cpu\"))\n\n    classifier = build_classifier(config).to(device)\n    classifier_path = run_dir / \"classifier.pt\"\n    if not classifier_path.exists():\n        classifier_path = resolve_path(config.get(\"feature_model\", {}).get(\"checkpoint_path\"), root)\n    _load_state_dict(classifier, classifier_path, device)\n    classifier.eval()\n    for p in classifier.parameters():\n        p.requires_grad_(False)\n\n    feature_model = build_feature_extractor(classifier, config).to(device)\n    feature_model.eval()\n\n    generator = build_generator(config).to(device)\n    generator_path = run_dir / \"generator_final.pt\"\n    _load_state_dict(generator, generator_path, device)\n\n    stats_path = run_dir / \"real_stats.pt\"\n    if not stats_path.exists():\n        stats_path = resolve_path(config.get(\"real_stats_path\"), root)\n    real_stats_obj = load_stats(stats_path, device=device)\n\n    eval_dir = run_dir / eval_config.get(\"outputs\", {}).get(\"eval_dir\", \"eval\")\n    return evaluate_generator_fresh(generator, classifier, feature_model, real_stats_obj, config, eval_config, eval_dir, device)\n",
  "src/fd_loss.py": "from __future__ import annotations\n\nimport torch\n\n\ndef lerp(a: float, b: float, t: float) -> float:\n    \"\"\"Linear interpolation.\"\"\"\n    return float(a + (b - a) * t)\n\n\ndef total_variation_loss(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Penalize pixel-to-pixel jumps.\"\"\"\n    tv_h = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()\n    tv_w = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()\n    return tv_h + tv_w\n\n\ndef foreground_mass_loss(x: torch.Tensor, target_mean: float = 0.13) -> torch.Tensor:\n    \"\"\"Keep generated MNIST foreground mass near a target average intensity.\"\"\"\n    return (x.mean() - target_mean).pow(2)\n\n\ndef border_loss(x: torch.Tensor, border: int = 3) -> torch.Tensor:\n    \"\"\"Penalize bright pixels near the image border.\"\"\"\n    top = x[:, :, :border, :].mean()\n    bottom = x[:, :, -border:, :].mean()\n    left = x[:, :, :, :border].mean()\n    right = x[:, :, :, -border:].mean()\n    return top + bottom + left + right\n\n\ndef sqrtm_psd(a: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:\n    \"\"\"Differentiable PSD matrix square root using eigenvalues.\"\"\"\n    a = 0.5 * (a + a.transpose(-1, -2))\n    eigvals, eigvecs = torch.linalg.eigh(a)\n    eigvals = eigvals.clamp_min(eps)\n    return (eigvecs * eigvals.sqrt().unsqueeze(-2)) @ eigvecs.transpose(-1, -2)\n\n\ndef batch_moments_by_class(\n    feats: torch.Tensor,\n    y: torch.Tensor,\n    num_classes: int,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    \"\"\"Return class-wise mean and raw second moment E[ff^T].\"\"\"\n    dim = feats.shape[1]\n    mus = []\n    m2s = []\n    for cls in range(num_classes):\n        f = feats[y == cls]\n        if f.shape[0] == 0:\n            mus.append(torch.zeros(dim, device=feats.device, dtype=feats.dtype))\n            m2s.append(torch.zeros(dim, dim, device=feats.device, dtype=feats.dtype))\n            continue\n        mus.append(f.mean(dim=0))\n        m2s.append(f.T @ f / f.shape[0])\n    return torch.stack(mus), torch.stack(m2s)\n\n\ndef cov_from_raw_moments(mu: torch.Tensor, m2: torch.Tensor, eps: float) -> torch.Tensor:\n    \"\"\"Convert mean and raw second moment to a regularized covariance.\"\"\"\n    dim = mu.shape[0]\n    cov = m2 - torch.outer(mu, mu)\n    cov = 0.5 * (cov + cov.T)\n    return cov + eps * torch.eye(dim, device=mu.device, dtype=mu.dtype)\n\n\ndef frechet_to_real(\n    mu_g: torch.Tensor,\n    cov_g: torch.Tensor,\n    real_layer_stats: dict[str, torch.Tensor],\n    cls: int,\n) -> dict[str, torch.Tensor]:\n    \"\"\"FD from generated Gaussian stats to fixed real stats for one layer/class.\"\"\"\n    mu_r = real_layer_stats[\"mu\"][cls]\n    cov_r = real_layer_stats[\"cov\"][cls]\n    cov_r_sqrt = real_layer_stats[\"cov_sqrt\"][cls]\n\n    mean_term = (mu_g - mu_r).pow(2).sum()\n    prod = cov_r_sqrt @ cov_g @ cov_r_sqrt\n    prod = 0.5 * (prod + prod.T)\n    eigvals = torch.linalg.eigvalsh(prod).clamp_min(1e-8)\n    trace_sqrt = eigvals.sqrt().sum()\n    cov_term = torch.trace(cov_g) + torch.trace(cov_r) - 2.0 * trace_sqrt\n    total = mean_term + cov_term\n    return {\"mean\": mean_term, \"cov\": cov_term, \"total\": total}\n\n\n@torch.no_grad()\ndef init_generated_ema(\n    generator,\n    feature_model,\n    layer_names: list[str],\n    layer_dims: dict[str, int],\n    num_classes: int,\n    z_dim: int,\n    device: torch.device,\n    rounds: int = 32,\n    per_class: int = 64,\n) -> dict[str, dict[str, torch.Tensor]]:\n    \"\"\"Warm-start generated EMA moments from the current generator.\"\"\"\n    generator.eval()\n    feature_model.eval()\n    buckets = {layer: [[] for _ in range(num_classes)] for layer in layer_names}\n\n    for _ in range(rounds):\n        y = torch.arange(num_classes, device=device).repeat_interleave(per_class)\n        z = torch.randn(y.numel(), z_dim, device=device)\n        x_fake = generator(z, y)\n        features = feature_model(x_fake)\n        for layer in layer_names:\n            feats = features[layer].detach()\n            for cls in range(num_classes):\n                buckets[layer][cls].append(feats[y == cls])\n\n    ema = {}\n    for layer in layer_names:\n        dim = layer_dims[layer]\n        mu = torch.zeros(num_classes, dim, device=device)\n        m2 = torch.zeros(num_classes, dim, dim, device=device)\n        for cls in range(num_classes):\n            f = torch.cat(buckets[layer][cls], dim=0)\n            mu[cls] = f.mean(dim=0)\n            m2[cls] = f.T @ f / f.shape[0]\n        ema[layer] = {\"mu\": mu, \"m2\": m2}\n    return ema\n\n\ndef layer_weighted_fd_from_ema_batch(\n    features: dict[str, torch.Tensor],\n    y: torch.Tensor,\n    ema_stats: dict[str, dict[str, torch.Tensor]],\n    real_stats: dict[str, dict[str, torch.Tensor]],\n    layer_weights: dict[str, float],\n    beta: float,\n    eps: float,\n    num_classes: int,\n    normalize: bool = True,\n) -> tuple[torch.Tensor, dict[str, float], dict[str, dict[str, torch.Tensor]]]:\n    \"\"\"Compute layer-weighted FD using detached EMA plus differentiable batch moments.\"\"\"\n    fd_total_norm = 0.0\n    fd_total_raw = 0.0\n    logs: dict[str, float] = {}\n    new_ema = {}\n\n    for layer, feats in features.items():\n        weight = float(layer_weights.get(layer, 1.0))\n        mu_b, m2_b = batch_moments_by_class(feats, y, num_classes)\n        layer_raw = 0.0\n        layer_norm = 0.0\n        layer_mean = 0.0\n        layer_cov = 0.0\n        new_mu = []\n        new_m2 = []\n\n        for cls in range(num_classes):\n            mu_g = beta * ema_stats[layer][\"mu\"][cls].detach() + (1.0 - beta) * mu_b[cls]\n            m2_g = beta * ema_stats[layer][\"m2\"][cls].detach() + (1.0 - beta) * m2_b[cls]\n            cov_g = cov_from_raw_moments(mu_g, m2_g, eps)\n            terms = frechet_to_real(mu_g, cov_g, real_stats[layer], cls)\n            fd = terms[\"total\"]\n\n            layer_raw = layer_raw + fd\n            layer_norm = layer_norm + fd / fd.detach().clamp_min(1e-6) if normalize else layer_norm + fd\n            layer_mean = layer_mean + terms[\"mean\"]\n            layer_cov = layer_cov + terms[\"cov\"]\n            new_mu.append(mu_g.detach())\n            new_m2.append(m2_g.detach())\n\n        layer_raw = layer_raw / num_classes\n        layer_norm = layer_norm / num_classes\n        layer_mean = layer_mean / num_classes\n        layer_cov = layer_cov / num_classes\n\n        fd_total_norm = fd_total_norm + weight * layer_norm\n        fd_total_raw = fd_total_raw + weight * layer_raw.detach()\n        logs[f\"fd_raw_{layer}\"] = float(layer_raw.detach().cpu())\n        logs[f\"fd_norm_{layer}\"] = float(layer_norm.detach().cpu())\n        logs[f\"fd_mean_{layer}\"] = float(layer_mean.detach().cpu())\n        logs[f\"fd_cov_{layer}\"] = float(layer_cov.detach().cpu())\n        new_ema[layer] = {\"mu\": torch.stack(new_mu), \"m2\": torch.stack(new_m2)}\n\n    logs[\"fd_total_raw\"] = float(fd_total_raw.detach().cpu())\n    logs[\"fd_total_norm\"] = float(fd_total_norm.detach().cpu())\n    return fd_total_norm, logs, new_ema\n",
  "src/features.py": "from __future__ import annotations\n\nfrom collections.abc import Callable\n\nimport torch\nimport torch.nn as nn\n\n\ndef identity(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Return x as a [B, D] feature tensor.\"\"\"\n    return x if x.ndim == 2 else x.flatten(1)\n\n\ndef channel_mean(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Channel means for [B, C, H, W] feature maps.\"\"\"\n    return x.mean(dim=(2, 3))\n\n\ndef channel_std(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Channel standard deviations for [B, C, H, W] feature maps.\"\"\"\n    return x.std(dim=(2, 3), unbiased=False)\n\n\ndef channel_mean_std(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Concatenate channel mean and channel std.\"\"\"\n    return torch.cat([channel_mean(x), channel_std(x)], dim=1)\n\n\ndef global_average_pool(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Global average pooling for image-like feature maps.\"\"\"\n    return x.mean(dim=(2, 3))\n\n\ndef flatten(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Flatten all non-batch dimensions.\"\"\"\n    return x.flatten(1)\n\n\ndef cls_token(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Return the first token from [B, T, D] features.\"\"\"\n    if x.ndim != 3:\n        raise ValueError(\"cls_token expects [B, T, D] features\")\n    return x[:, 0]\n\n\ndef token_mean(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Mean over tokens for [B, T, D] features.\"\"\"\n    if x.ndim != 3:\n        raise ValueError(\"token_mean expects [B, T, D] features\")\n    return x.mean(dim=1)\n\n\ndef token_mean_std(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Concatenate token mean and token std for [B, T, D] features.\"\"\"\n    if x.ndim != 3:\n        raise ValueError(\"token_mean_std expects [B, T, D] features\")\n    return torch.cat([x.mean(dim=1), x.std(dim=1, unbiased=False)], dim=1)\n\n\nREDUCERS: dict[str, Callable[[torch.Tensor], torch.Tensor]] = {\n    \"identity\": identity,\n    \"channel_mean\": channel_mean,\n    \"channel_std\": channel_std,\n    \"channel_mean_std\": channel_mean_std,\n    \"global_average_pool\": global_average_pool,\n    \"flatten\": flatten,\n    \"cls_token\": cls_token,\n    \"token_mean\": token_mean,\n    \"token_mean_std\": token_mean_std,\n}\n\n\nclass ConfiguredFeatureExtractor(nn.Module):\n    \"\"\"Generic dict[str, Tensor] feature extractor driven by layer specs.\"\"\"\n\n    def __init__(self, model: nn.Module, layer_specs: list[dict]):\n        super().__init__()\n        self.model = model\n        self.layer_specs = layer_specs\n\n    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:\n        if hasattr(self.model, \"forward_feature_sources\"):\n            sources = self.model.forward_feature_sources(x)\n        else:\n            sources = self.model(x)\n            if not isinstance(sources, dict):\n                raise TypeError(\"Feature model must return dict[str, Tensor] or implement forward_feature_sources\")\n\n        out = {}\n        for spec in self.layer_specs:\n            name = spec[\"name\"]\n            source_name = spec.get(\"source\", name)\n            reducer_name = spec.get(\"reducer\", \"identity\")\n            if source_name not in sources:\n                raise KeyError(f\"Feature source {source_name!r} not found. Available: {list(sources)}\")\n            if reducer_name not in REDUCERS:\n                raise KeyError(f\"Unknown reducer {reducer_name!r}. Available: {list(REDUCERS)}\")\n            out[name] = REDUCERS[reducer_name](sources[source_name])\n        return out\n\n\ndef get_layer_specs(config: dict) -> list[dict]:\n    \"\"\"Return layer specs from a config.\"\"\"\n    specs = config.get(\"features\", {}).get(\"selected_layers\", [])\n    if not specs:\n        raise ValueError(\"Config must define features.selected_layers\")\n    return specs\n\n\ndef get_layer_weights(config: dict) -> dict[str, float]:\n    \"\"\"Return layer weights keyed by layer name.\"\"\"\n    return {spec[\"name\"]: float(spec.get(\"weight\", 1.0)) for spec in get_layer_specs(config)}\n\n\n@torch.no_grad()\ndef infer_layer_dims(feature_model: nn.Module, sample: torch.Tensor, device: torch.device) -> dict[str, int]:\n    \"\"\"Infer feature dimensions by running one sample batch.\"\"\"\n    feature_model.eval()\n    feats = feature_model(sample.to(device))\n    return {name: int(value.shape[1]) for name, value in feats.items()}\n\n\ndef build_feature_extractor(model: nn.Module, config: dict) -> ConfiguredFeatureExtractor:\n    \"\"\"Build a configured feature extractor around a model.\"\"\"\n    return ConfiguredFeatureExtractor(model, get_layer_specs(config))\n",
  "src/models.py": "from __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass MNISTClassifier(nn.Module):\n    \"\"\"Small CNN classifier exposing named intermediate feature sources.\"\"\"\n\n    def __init__(self, num_classes: int = 10):\n        super().__init__()\n        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)\n        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)\n        self.fc1 = nn.Linear(64 * 7 * 7, 128)\n        self.dropout = nn.Dropout(0.10)\n        self.head = nn.Linear(128, num_classes)\n\n    def forward_feature_sources(self, x: torch.Tensor) -> dict[str, torch.Tensor]:\n        \"\"\"Return reusable feature sources before reducer selection.\"\"\"\n        conv1 = F.relu(self.conv1(x))\n        pool1 = F.max_pool2d(conv1, 2)\n        conv2 = F.relu(self.conv2(pool1))\n        pool2 = F.max_pool2d(conv2, 2)\n        late = F.relu(self.fc1(pool2.flatten(1)))\n        return {\n            \"conv1\": conv1,\n            \"pool1\": pool1,\n            \"conv2\": conv2,\n            \"pool2\": pool2,\n            \"late\": late,\n        }\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        sources = self.forward_feature_sources(x)\n        return self.head(self.dropout(sources[\"late\"]))\n\n\nclass OneStepGenerator(nn.Module):\n    \"\"\"Simple conditional one-step MNIST generator.\"\"\"\n\n    def __init__(self, z_dim: int = 64, num_classes: int = 10, image_shape=(1, 28, 28)):\n        super().__init__()\n        self.z_dim = int(z_dim)\n        self.num_classes = int(num_classes)\n        self.image_shape = tuple(image_shape)\n        out_dim = int(torch.tensor(image_shape).prod().item())\n        self.y_emb = nn.Embedding(num_classes, 16)\n        self.net = nn.Sequential(\n            nn.Linear(self.z_dim + 16, 256),\n            nn.SiLU(),\n            nn.Linear(256, 512),\n            nn.SiLU(),\n            nn.Linear(512, 1024),\n            nn.SiLU(),\n            nn.Linear(1024, out_dim),\n        )\n\n    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n        y_e = self.y_emb(y)\n        h = torch.cat([z, y_e], dim=1)\n        x = torch.sigmoid(self.net(h))\n        return x.view(-1, *self.image_shape)\n\n\ndef build_classifier(config: dict) -> nn.Module:\n    \"\"\"Classifier factory.\"\"\"\n    cls_cfg = config.get(\"classifier\", config.get(\"feature_model\", {}))\n    model_type = cls_cfg.get(\"type\", \"mnist_cnn\")\n    num_classes = int(config.get(\"num_classes\", 10))\n    if model_type != \"mnist_cnn\":\n        raise ValueError(f\"Unsupported classifier type: {model_type}\")\n    return MNISTClassifier(num_classes=num_classes)\n\n\ndef build_generator(config: dict) -> nn.Module:\n    \"\"\"Generator factory.\"\"\"\n    gen_cfg = config.get(\"generator\", {})\n    model_type = gen_cfg.get(\"type\", \"one_step_mnist\")\n    num_classes = int(config.get(\"num_classes\", 10))\n    if model_type != \"one_step_mnist\":\n        raise ValueError(f\"Unsupported generator type: {model_type}\")\n    return OneStepGenerator(\n        z_dim=int(gen_cfg.get(\"z_dim\", 64)),\n        num_classes=num_classes,\n        image_shape=tuple(gen_cfg.get(\"image_shape\", (1, 28, 28))),\n    )\n\n\ndef augment_mnist_batch(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"Small random translations plus mild noise.\"\"\"\n    shifts_y = torch.randint(-2, 3, (x.size(0),), device=x.device)\n    shifts_x = torch.randint(-2, 3, (x.size(0),), device=x.device)\n    x_aug = x.clone()\n    for i in range(x.size(0)):\n        x_aug[i] = torch.roll(x_aug[i], shifts=(int(shifts_y[i]), int(shifts_x[i])), dims=(1, 2))\n    x_aug = x_aug + 0.03 * torch.randn_like(x_aug)\n    return x_aug.clamp(0.0, 1.0)\n\n\n@torch.no_grad()\ndef evaluate_classifier(model: nn.Module, loader, device: torch.device) -> dict[str, float]:\n    \"\"\"Return loss and accuracy for a classifier.\"\"\"\n    model.eval()\n    total_loss = 0.0\n    correct = 0\n    total = 0\n    for xb, yb in loader:\n        xb, yb = xb.to(device), yb.to(device)\n        logits = model(xb)\n        loss = F.cross_entropy(logits, yb)\n        total_loss += loss.item() * xb.size(0)\n        correct += (logits.argmax(dim=1) == yb).sum().item()\n        total += xb.size(0)\n    return {\"loss\": total_loss / total, \"acc\": correct / total}\n",
  "src/plotting.py": "from __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\n\n\ndef to_numpy(x) -> np.ndarray:\n    \"\"\"Convert tensor-like values to NumPy.\"\"\"\n    if torch.is_tensor(x):\n        return x.detach().cpu().numpy()\n    return np.asarray(x)\n\n\ndef covariance_to_correlation(cov) -> np.ndarray:\n    \"\"\"Convert covariance to correlation with numerical safety.\"\"\"\n    cov = to_numpy(cov).astype(float)\n    diag = np.sqrt(np.clip(np.diag(cov), 1e-12, None))\n    corr = cov / np.outer(diag, diag)\n    return np.clip(corr, -1.0, 1.0)\n\n\ndef percentile_limits(mat, percentile: float = 99.0, symmetric: bool = False):\n    \"\"\"Return display limits using percentile clipping.\"\"\"\n    mat = to_numpy(mat)\n    if symmetric:\n        vmax = np.nanpercentile(np.abs(mat), percentile)\n        return -float(vmax), float(vmax)\n    vmin, vmax = np.nanpercentile(mat, [100 - percentile, percentile])\n    return float(vmin), float(vmax)\n\n\ndef cluster_order(corr, method: str = \"average\") -> np.ndarray:\n    \"\"\"Hierarchical order for a correlation matrix; falls back to original order.\"\"\"\n    corr = to_numpy(corr)\n    try:\n        from scipy.cluster.hierarchy import leaves_list, linkage\n        from scipy.spatial.distance import squareform\n\n        dist = 1.0 - np.abs(np.nan_to_num(corr, nan=0.0))\n        dist = 0.5 * (dist + dist.T)\n        np.fill_diagonal(dist, 0.0)\n        return leaves_list(linkage(squareform(dist, checks=False), method=method))\n    except Exception:\n        return np.arange(corr.shape[0])\n\n\ndef variance_order(cov) -> np.ndarray:\n    \"\"\"Feature order sorted by descending variance.\"\"\"\n    cov = to_numpy(cov)\n    return np.argsort(-np.diag(cov))\n\n\ndef apply_order(mat, order=None):\n    \"\"\"Apply the same row/column order to a square matrix.\"\"\"\n    mat = to_numpy(mat)\n    if order is None:\n        return mat\n    return mat[np.ix_(order, order)]\n\n\ndef save_fig(fig, path: str | Path | None):\n    \"\"\"Save and close a figure when path is provided.\"\"\"\n    if path is not None:\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(path, dpi=170, bbox_inches=\"tight\")\n    plt.close(fig)\n\n\ndef plot_matrix(mat, title: str, path: str | Path | None = None, cmap=\"viridis\", vmin=None, vmax=None):\n    \"\"\"Plot one matrix.\"\"\"\n    fig, ax = plt.subplots(figsize=(6, 5))\n    im = ax.imshow(to_numpy(mat), cmap=cmap, vmin=vmin, vmax=vmax)\n    ax.set_title(title)\n    ax.set_xticks([])\n    ax.set_yticks([])\n    fig.colorbar(im, ax=ax, shrink=0.8)\n    save_fig(fig, path)\n    return fig\n\n\ndef plot_covariance_matrix(cov, title=\"Covariance\", path=None, percentile: float = 99.0):\n    \"\"\"Plot a covariance matrix with percentile clipping.\"\"\"\n    vmin, vmax = percentile_limits(cov, percentile=percentile, symmetric=False)\n    return plot_matrix(cov, title=title, path=path, cmap=\"viridis\", vmin=vmin, vmax=vmax)\n\n\ndef plot_correlation_matrix(corr, title=\"Correlation\", path=None, percentile: float = 99.0):\n    \"\"\"Plot a correlation matrix with symmetric color limits.\"\"\"\n    vmin, vmax = percentile_limits(corr, percentile=percentile, symmetric=True)\n    return plot_matrix(corr, title=title, path=path, cmap=\"coolwarm\", vmin=vmin, vmax=vmax)\n\n\ndef plot_clustered_correlation(cov_or_corr, title=\"Clustered correlation\", path=None, percentile: float = 99.0):\n    \"\"\"Plot correlation ordered by hierarchical clustering.\"\"\"\n    corr = covariance_to_correlation(cov_or_corr)\n    order = cluster_order(corr)\n    return plot_correlation_matrix(apply_order(corr, order), title=title, path=path, percentile=percentile)\n\n\ndef plot_triplet(real, generated, title: str, path=None, cmap=\"viridis\", percentile: float = 99.0, signed=False):\n    \"\"\"Plot real, generated, and generated-real using one order/scale.\"\"\"\n    real = to_numpy(real)\n    generated = to_numpy(generated)\n    diff = generated - real\n    if signed:\n        vmin, vmax = percentile_limits(np.concatenate([real.ravel(), generated.ravel()]), percentile, symmetric=True)\n        dvmin, dvmax = percentile_limits(diff, percentile, symmetric=True)\n        cmap_main = cmap\n        cmap_diff = \"coolwarm\"\n    else:\n        vmin, vmax = percentile_limits(np.concatenate([real.ravel(), generated.ravel()]), percentile, symmetric=False)\n        dvmin, dvmax = percentile_limits(diff, percentile, symmetric=True)\n        cmap_main = cmap\n        cmap_diff = \"coolwarm\"\n\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))\n    for ax, mat, subtitle, cm, lo, hi in [\n        (axes[0], real, \"real\", cmap_main, vmin, vmax),\n        (axes[1], generated, \"generated\", cmap_main, vmin, vmax),\n        (axes[2], diff, \"generated - real\", cmap_diff, dvmin, dvmax),\n    ]:\n        im = ax.imshow(mat, cmap=cm, vmin=lo, vmax=hi)\n        ax.set_title(subtitle)\n        ax.set_xticks([])\n        ax.set_yticks([])\n        fig.colorbar(im, ax=ax, shrink=0.75)\n    fig.suptitle(title)\n    save_fig(fig, path)\n    return fig\n\n\ndef plot_real_generated_covariance_triplet(real_cov, gen_cov, title, path=None, percentile: float = 99.0):\n    \"\"\"Plot real/generated/difference covariance matrices.\"\"\"\n    return plot_triplet(real_cov, gen_cov, title=title, path=path, cmap=\"viridis\", percentile=percentile)\n\n\ndef plot_real_generated_correlation_triplet(real_cov, gen_cov, title, path=None, percentile: float = 99.0, order=None):\n    \"\"\"Plot real/generated/difference correlations with one shared ordering.\"\"\"\n    real_corr = covariance_to_correlation(real_cov)\n    gen_corr = covariance_to_correlation(gen_cov)\n    if order is None:\n        order = cluster_order(real_corr)\n    real_corr = apply_order(real_corr, order)\n    gen_corr = apply_order(gen_corr, order)\n    return plot_triplet(real_corr, gen_corr, title=title, path=path, cmap=\"coolwarm\", percentile=percentile, signed=True)\n\n\ndef plot_eigenspectrum(covariances: dict[str, np.ndarray], title: str, path=None):\n    \"\"\"Plot eigenvalue spectra for one or more covariance matrices.\"\"\"\n    fig, ax = plt.subplots(figsize=(7, 4.5))\n    for label, cov in covariances.items():\n        vals = np.linalg.eigvalsh(to_numpy(cov))\n        vals = np.sort(np.clip(vals, 0.0, None))[::-1]\n        ax.plot(np.arange(1, len(vals) + 1), vals, label=label)\n    ax.set_title(title)\n    ax.set_xlabel(\"eigenvalue index\")\n    ax.set_ylabel(\"eigenvalue\")\n    ax.set_yscale(\"log\")\n    ax.grid(alpha=0.25)\n    ax.legend()\n    save_fig(fig, path)\n    return fig\n\n\ndef plot_sample_grid(images, labels=None, n_per_class: int = 10, path=None, title=\"Samples\"):\n    \"\"\"Plot a class-organized sample grid.\"\"\"\n    images = to_numpy(images)\n    if labels is None:\n        labels = np.repeat(np.arange(images.shape[0] // n_per_class), n_per_class)\n    labels = to_numpy(labels).astype(int)\n    classes = list(dict.fromkeys(labels.tolist()))\n    rows = len(classes)\n    fig, axes = plt.subplots(rows, n_per_class, figsize=(n_per_class, max(rows, 1)))\n    axes = np.asarray(axes).reshape(rows, n_per_class)\n    for r, cls in enumerate(classes):\n        idxs = np.where(labels == cls)[0][:n_per_class]\n        for c in range(n_per_class):\n            ax = axes[r, c]\n            ax.axis(\"off\")\n            if c < len(idxs):\n                img = images[idxs[c], 0]\n                ax.imshow(img, cmap=\"gray\", vmin=0, vmax=1)\n            if c == 0:\n                ax.set_ylabel(str(cls), rotation=0, labelpad=12)\n    fig.suptitle(title)\n    save_fig(fig, path)\n    return fig\n\n\ndef plot_sample_grid_with_predictions(images, labels, preds, n_per_class: int = 10, path=None):\n    \"\"\"Plot sample grid with classifier predictions as titles.\"\"\"\n    images = to_numpy(images)\n    labels = to_numpy(labels).astype(int)\n    preds = to_numpy(preds).astype(int)\n    classes = list(dict.fromkeys(labels.tolist()))\n    rows = len(classes)\n    fig, axes = plt.subplots(rows, n_per_class, figsize=(n_per_class, max(rows, 1)))\n    axes = np.asarray(axes).reshape(rows, n_per_class)\n    for r, cls in enumerate(classes):\n        idxs = np.where(labels == cls)[0][:n_per_class]\n        for c in range(n_per_class):\n            ax = axes[r, c]\n            ax.axis(\"off\")\n            if c < len(idxs):\n                idx = idxs[c]\n                ax.imshow(images[idx, 0], cmap=\"gray\", vmin=0, vmax=1)\n                ax.set_title(str(preds[idx]), fontsize=8)\n            if c == 0:\n                ax.set_ylabel(f\"y={cls}\", rotation=0, labelpad=12)\n    save_fig(fig, path)\n    return fig\n\n\ndef plot_metrics_curves(metrics_csv: str | Path, path: str | Path | None = None):\n    \"\"\"Plot common training curves from metrics.csv.\"\"\"\n    df = pd.read_csv(metrics_csv)\n    x = df[\"step\"] if \"step\" in df else np.arange(len(df))\n    fig, axes = plt.subplots(3, 2, figsize=(15, 12))\n    axes = axes.ravel()\n\n    groups = [\n        (\"Loss\", [\"total_loss\", \"ce_loss\", \"fd_total_norm\", \"fd_total_raw\"]),\n        (\"FD by layer\", [c for c in df.columns if c.startswith(\"fd_raw_\")]),\n        (\"FD decomposition\", [c for c in df.columns if c.startswith(\"fd_mean_\") or c.startswith(\"fd_cov_\")]),\n        (\"Image priors\", [\"tv_loss\", \"foreground_mass_loss\", \"border_loss\"]),\n        (\"Schedules\", [\"beta\", \"lambda_fd\", \"lambda_ce\", \"lambda_tv\", \"lambda_foreground_mass\", \"lambda_border\"]),\n        (\"Diagnostics\", [\"classifier_fake_acc\", \"x_mean\", \"x_std\", \"grad_norm\", \"lr\"]),\n    ]\n    for ax, (title, cols) in zip(axes, groups):\n        for col in cols:\n            if col in df:\n                ax.plot(x, df[col], label=col)\n        ax.set_title(title)\n        ax.set_xlabel(\"step\")\n        ax.grid(alpha=0.25)\n        if ax.lines:\n            ax.legend(fontsize=8)\n    fig.tight_layout()\n    save_fig(fig, path)\n    return fig\n",
  "src/stats.py": "from __future__ import annotations\n\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\n\n\ndef sqrtm_psd_nograd(a: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:\n    \"\"\"PSD matrix square root without gradients.\"\"\"\n    a = 0.5 * (a + a.transpose(-1, -2))\n    eigvals, eigvecs = torch.linalg.eigh(a)\n    eigvals = eigvals.clamp_min(eps)\n    return (eigvecs * eigvals.sqrt().unsqueeze(-2)) @ eigvecs.transpose(-1, -2)\n\n\ndef compute_mean_cov_from_sums(\n    n: torch.Tensor,\n    sums: torch.Tensor,\n    sums_outer: torch.Tensor,\n    eps: float,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    \"\"\"Convert class-wise sums into means and regularized covariance matrices.\"\"\"\n    num_classes, dim = sums.shape\n    mu = torch.zeros(num_classes, dim, dtype=sums.dtype)\n    cov = torch.zeros(num_classes, dim, dim, dtype=sums.dtype)\n    eye = torch.eye(dim, dtype=sums.dtype)\n\n    for cls in range(num_classes):\n        count = int(n[cls].item())\n        if count == 0:\n            cov[cls] = eye * eps\n            continue\n        mu[cls] = sums[cls] / count\n        centered_outer = sums_outer[cls] - count * torch.outer(mu[cls], mu[cls])\n        denom = max(count - 1, 1)\n        cov_cls = centered_outer / denom\n        cov_cls = 0.5 * (cov_cls + cov_cls.T)\n        cov[cls] = cov_cls + eps * eye\n    return mu, cov\n\n\n@torch.no_grad()\ndef compute_feature_stats(\n    feature_model,\n    loader,\n    num_classes: int,\n    device: torch.device,\n    eps: float = 1e-4,\n) -> dict[str, dict[str, torch.Tensor]]:\n    \"\"\"Compute per-layer, per-class mean/cov/cov_sqrt/n from a frozen feature model.\"\"\"\n    feature_model.eval()\n    accum: dict[str, dict[str, torch.Tensor]] = {}\n\n    for xb, yb in loader:\n        xb = xb.to(device)\n        yb = yb.cpu()\n        features = {k: v.detach().cpu().float() for k, v in feature_model(xb).items()}\n\n        for layer, feats in features.items():\n            dim = feats.shape[1]\n            if layer not in accum:\n                accum[layer] = {\n                    \"n\": torch.zeros(num_classes, dtype=torch.long),\n                    \"sum\": torch.zeros(num_classes, dim),\n                    \"outer\": torch.zeros(num_classes, dim, dim),\n                }\n            for cls in range(num_classes):\n                f = feats[yb == cls]\n                if f.numel() == 0:\n                    continue\n                accum[layer][\"n\"][cls] += f.shape[0]\n                accum[layer][\"sum\"][cls] += f.sum(dim=0)\n                accum[layer][\"outer\"][cls] += f.T @ f\n\n    stats = {}\n    for layer, layer_accum in accum.items():\n        mu, cov = compute_mean_cov_from_sums(\n            layer_accum[\"n\"],\n            layer_accum[\"sum\"],\n            layer_accum[\"outer\"],\n            eps=eps,\n        )\n        cov_sqrt = torch.stack([sqrtm_psd_nograd(cov[cls]) for cls in range(num_classes)])\n        stats[layer] = {\n            \"mu\": mu,\n            \"cov\": cov,\n            \"cov_sqrt\": cov_sqrt,\n            \"n\": layer_accum[\"n\"],\n            \"dim\": torch.tensor(mu.shape[1]),\n        }\n    return stats\n\n\ndef stats_metadata(\n    dataset_name: str,\n    num_classes: int,\n    feature_config: dict[str, Any],\n    classifier_checkpoint: str | Path | None,\n    epsilon: float,\n    stats: dict[str, dict[str, torch.Tensor]],\n    run_id: str | None = None,\n) -> dict[str, Any]:\n    \"\"\"Build metadata for a saved real_stats.pt artifact.\"\"\"\n    layer_names = list(stats.keys())\n    return {\n        \"dataset\": dataset_name,\n        \"num_classes\": int(num_classes),\n        \"layer_names\": layer_names,\n        \"layer_dims\": {layer: int(stats[layer][\"dim\"].item()) for layer in layer_names},\n        \"feature_config\": feature_config,\n        \"feature_model_checkpoint\": str(classifier_checkpoint) if classifier_checkpoint else None,\n        \"covariance_epsilon\": float(epsilon),\n        \"examples_per_class\": {layer: stats[layer][\"n\"].tolist() for layer in layer_names},\n        \"created_at\": datetime.now().isoformat(timespec=\"seconds\"),\n        \"run_id\": run_id,\n    }\n\n\ndef save_stats(stats: dict, metadata: dict[str, Any], path: str | Path) -> None:\n    \"\"\"Save stats and metadata to a Torch artifact.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save({\"stats\": stats, \"metadata\": metadata}, path)\n\n\ndef load_stats(path: str | Path, device: torch.device | str | None = None) -> dict:\n    \"\"\"Load stats artifact and optionally move tensors to a device.\"\"\"\n    obj = torch.load(path, map_location=\"cpu\")\n    if device is None:\n        return obj\n    device = torch.device(device)\n    for layer, layer_stats in obj[\"stats\"].items():\n        for key, value in list(layer_stats.items()):\n            if torch.is_tensor(value):\n                layer_stats[key] = value.to(device)\n    return obj\n\n\ndef layer_names_from_stats(stats_obj: dict) -> list[str]:\n    \"\"\"Return layer names from a stats artifact or plain stats dict.\"\"\"\n    if \"metadata\" in stats_obj:\n        return list(stats_obj[\"metadata\"].get(\"layer_names\", stats_obj[\"stats\"].keys()))\n    return list(stats_obj.keys())\n",
  "src/utils.py": "from __future__ import annotations\n\nimport csv\nimport json\nimport random\nimport shutil\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\n\n\ndef load_config(path: str | Path) -> dict[str, Any]:\n    \"\"\"Load a YAML or JSON config file.\"\"\"\n    path = Path(path)\n    with path.open(\"r\", encoding=\"utf-8\") as f:\n        if path.suffix.lower() in {\".yaml\", \".yml\"}:\n            import yaml\n\n            return yaml.safe_load(f)\n        return json.load(f)\n\n\ndef save_config(config: dict[str, Any], path: str | Path) -> None:\n    \"\"\"Save a config as YAML.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        try:\n            import yaml\n\n            yaml.safe_dump(config, f, sort_keys=False)\n        except ModuleNotFoundError:\n            json.dump(config, f, indent=2)\n\n\ndef save_json(data: dict[str, Any], path: str | Path) -> None:\n    \"\"\"Save JSON with stable indentation.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        json.dump(data, f, indent=2)\n\n\ndef resolve_path(path: str | Path | None, root: str | Path = PROJECT_ROOT) -> Path | None:\n    \"\"\"Resolve a possibly relative path against the project root.\"\"\"\n    if path is None or path == \"\":\n        return None\n    path = Path(path)\n    if path.is_absolute():\n        return path\n    return Path(root) / path\n\n\ndef set_seed(seed: int | None) -> None:\n    \"\"\"Seed Python, NumPy, and Torch.\"\"\"\n    if seed is None:\n        return\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef get_device(device: str | None = \"auto\") -> torch.device:\n    \"\"\"Resolve a config device string.\"\"\"\n    if device in (None, \"auto\"):\n        return torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    return torch.device(device)\n\n\ndef make_run_dir(run_root: str | Path, run_name: str, root: str | Path = PROJECT_ROOT) -> Path:\n    \"\"\"Create runs/<timestamp>_<run_name> and return it.\"\"\"\n    run_root = resolve_path(run_root, root)\n    run_root.mkdir(parents=True, exist_ok=True)\n    safe_name = \"\".join(c if c.isalnum() or c in \"-_\" else \"_\" for c in run_name)\n    stamp = datetime.now().strftime(\"%Y%m%d-%H%M%S\")\n    run_dir = run_root / f\"{stamp}_{safe_name}\"\n    suffix = 1\n    while run_dir.exists():\n        run_dir = run_root / f\"{stamp}_{safe_name}_{suffix:02d}\"\n        suffix += 1\n    (run_dir / \"samples\").mkdir(parents=True, exist_ok=True)\n    (run_dir / \"plots\").mkdir(parents=True, exist_ok=True)\n    return run_dir\n\n\ndef save_metrics_csv(rows: list[dict[str, Any]], path: str | Path) -> None:\n    \"\"\"Write scalar metric rows to CSV.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        return\n    fieldnames = sorted({key for row in rows for key in row.keys()})\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef copy_if_exists(src: str | Path | None, dst: str | Path) -> bool:\n    \"\"\"Copy an artifact if it exists.\"\"\"\n    if src is None:\n        return False\n    src = Path(src)\n    if not src.exists():\n        return False\n    dst = Path(dst)\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(src, dst)\n    return True\n\n\ndef get_lr(optimizer: torch.optim.Optimizer) -> float:\n    \"\"\"Return the first param-group learning rate.\"\"\"\n    return float(optimizer.param_groups[0][\"lr\"])\n\n\ndef to_float(x: Any) -> float:\n    \"\"\"Convert tensors and NumPy scalars to plain Python floats.\"\"\"\n    if torch.is_tensor(x):\n        return float(x.detach().cpu())\n    return float(x)\n",
  "scripts/compute_real_stats.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\n\nfrom src.data import load_data\nfrom src.features import build_feature_extractor\nfrom src.models import build_classifier\nfrom src.stats import compute_feature_stats, save_stats, stats_metadata\nfrom src.utils import get_device, load_config, resolve_path, set_seed\n\n\ndef load_model_state(model, checkpoint_path: Path, device: torch.device) -> None:\n    obj = torch.load(checkpoint_path, map_location=device)\n    state = obj.get(\"model_state_dict\", obj) if isinstance(obj, dict) else obj\n    model.load_state_dict(state)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=ROOT / \"configs\" / \"classifier_mnist.yaml\")\n    args = parser.parse_args()\n\n    config = load_config(args.config)\n    set_seed(config.get(\"seed\"))\n    device = get_device(config.get(\"device\", \"auto\"))\n    data = load_data(config, root=ROOT)\n\n    classifier = build_classifier(config).to(device)\n    ckpt_path = resolve_path(config[\"classifier\"][\"checkpoint_path\"], ROOT)\n    load_model_state(classifier, ckpt_path, device)\n    classifier.eval()\n    for p in classifier.parameters():\n        p.requires_grad_(False)\n\n    feature_model = build_feature_extractor(classifier, config).to(device)\n    eps = float(config.get(\"stats\", {}).get(\"epsilon\", 1e-4))\n    stats = compute_feature_stats(feature_model, data.train_loader, data.num_classes, device, eps=eps)\n    metadata = stats_metadata(\n        dataset_name=data.name,\n        num_classes=data.num_classes,\n        feature_config=config.get(\"features\", {}),\n        classifier_checkpoint=ckpt_path,\n        epsilon=eps,\n        stats=stats,\n        run_id=\"real_stats\",\n    )\n    out_path = resolve_path(config.get(\"stats\", {}).get(\"output_path\", \"runs/classifier_mnist/real_stats.pt\"), ROOT)\n    save_stats(stats, metadata, out_path)\n    print(f\"saved real feature stats to {out_path}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/evaluate_generator.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\n\nfrom src.eval import run_evaluation\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--run\", required=True, help=\"Run folder, for example runs/20260508-120000_layer_fd_test\")\n    parser.add_argument(\"--config\", default=ROOT / \"configs\" / \"eval_mnist.yaml\")\n    args = parser.parse_args()\n    summary = run_evaluation(args.run, args.config, root=ROOT)\n    print(\"evaluation summary:\")\n    for key, value in summary.items():\n        if key == \"confusion_matrix\":\n            continue\n        print(f\"  {key}: {value}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/train_classifier.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport torch\nimport torch.nn.functional as F\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\n\nfrom src.data import load_data\nfrom src.models import augment_mnist_batch, build_classifier, evaluate_classifier\nfrom src.utils import get_device, load_config, resolve_path, save_metrics_csv, set_seed\n\n\ndef plot_classifier_history(rows, path: Path) -> None:\n    \"\"\"Save train/eval curves for classifier pretraining.\"\"\"\n    if not rows:\n        return\n    epochs = [row[\"epoch\"] for row in rows]\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    axes[0].plot(epochs, [row[\"train_loss\"] for row in rows], label=\"train\")\n    axes[0].plot(epochs, [row[\"eval_loss\"] for row in rows], label=\"eval\")\n    axes[0].set_title(\"Classifier loss\")\n    axes[0].set_xlabel(\"epoch\")\n    axes[0].grid(alpha=0.25)\n    axes[0].legend()\n    axes[1].plot(epochs, [row[\"train_acc\"] for row in rows], label=\"train\")\n    axes[1].plot(epochs, [row[\"eval_acc\"] for row in rows], label=\"eval\")\n    axes[1].set_title(\"Classifier accuracy\")\n    axes[1].set_xlabel(\"epoch\")\n    axes[1].set_ylim(0, 1)\n    axes[1].grid(alpha=0.25)\n    axes[1].legend()\n    fig.tight_layout()\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=170)\n    plt.close(fig)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=ROOT / \"configs\" / \"classifier_mnist.yaml\")\n    args = parser.parse_args()\n\n    config = load_config(args.config)\n    set_seed(config.get(\"seed\"))\n    device = get_device(config.get(\"device\", \"auto\"))\n    data = load_data(config, root=ROOT)\n    model = build_classifier(config).to(device)\n    cls_cfg = config[\"classifier\"]\n\n    opt = torch.optim.AdamW(\n        model.parameters(),\n        lr=float(cls_cfg.get(\"lr\", 1e-3)),\n        weight_decay=float(cls_cfg.get(\"weight_decay\", 0.0)),\n    )\n    epochs = int(cls_cfg.get(\"epochs\", 10))\n    use_aug = bool(cls_cfg.get(\"augment\", True))\n    rows = []\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        total_loss = 0.0\n        correct = 0\n        total = 0\n        for xb, yb in data.train_loader:\n            xb, yb = xb.to(device), yb.to(device)\n            if use_aug:\n                xb = augment_mnist_batch(xb)\n            logits = model(xb)\n            loss = F.cross_entropy(logits, yb)\n            opt.zero_grad(set_to_none=True)\n            loss.backward()\n            opt.step()\n\n            total_loss += loss.item() * xb.size(0)\n            correct += (logits.argmax(dim=1) == yb).sum().item()\n            total += xb.size(0)\n\n        eval_metrics = evaluate_classifier(model, data.test_loader, device)\n        row = {\n            \"epoch\": epoch,\n            \"train_loss\": total_loss / total,\n            \"train_acc\": correct / total,\n            \"eval_loss\": eval_metrics[\"loss\"],\n            \"eval_acc\": eval_metrics[\"acc\"],\n        }\n        rows.append(row)\n        print(\n            f\"epoch {epoch:03d}/{epochs:03d} | \"\n            f\"train_loss={row['train_loss']:.4f} train_acc={row['train_acc']:.4f} | \"\n            f\"eval_loss={row['eval_loss']:.4f} eval_acc={row['eval_acc']:.4f}\"\n        )\n\n    ckpt_path = resolve_path(cls_cfg[\"checkpoint_path\"], ROOT)\n    ckpt_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save({\"model_state_dict\": model.state_dict(), \"config\": config}, ckpt_path)\n\n    history_path = resolve_path(cls_cfg.get(\"history_path\"), ROOT)\n    curves_path = resolve_path(cls_cfg.get(\"curves_path\"), ROOT)\n    if history_path:\n        save_metrics_csv(rows, history_path)\n    if curves_path:\n        plot_classifier_history(rows, curves_path)\n    print(f\"saved classifier checkpoint to {ckpt_path}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/train_generator_fd.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\n\nfrom src.data import load_data\nfrom src.fd_loss import (\n    border_loss,\n    cov_from_raw_moments,\n    foreground_mass_loss,\n    frechet_to_real,\n    init_generated_ema,\n    layer_weighted_fd_from_ema_batch,\n    lerp,\n    total_variation_loss,\n)\nfrom src.features import build_feature_extractor, get_layer_weights\nfrom src.models import build_classifier, build_generator\nfrom src.plotting import plot_metrics_curves, plot_sample_grid_with_predictions\nfrom src.stats import load_stats\nfrom src.utils import (\n    copy_if_exists,\n    get_device,\n    get_lr,\n    load_config,\n    make_run_dir,\n    resolve_path,\n    save_config,\n    save_json,\n    save_metrics_csv,\n    set_seed,\n)\n\n\ndef load_state_dict(model, checkpoint_path: Path, device: torch.device) -> None:\n    obj = torch.load(checkpoint_path, map_location=device)\n    state = obj.get(\"model_state_dict\", obj) if isinstance(obj, dict) else obj\n    model.load_state_dict(state)\n\n\ndef current_loss_weights(step: int, config: dict) -> dict[str, float]:\n    \"\"\"Return warmup or main loss weights for the current step.\"\"\"\n    lw = config[\"loss_weights\"]\n    phase = \"warmup\" if step <= int(lw.get(\"ce_warmup_steps\", 0)) else \"main\"\n    values = lw[phase]\n    return {\n        \"fd\": float(values.get(\"fd\", 1.0)),\n        \"ce\": float(values.get(\"ce\", 0.0)),\n        \"tv\": float(values.get(\"tv\", 0.0)),\n        \"foreground_mass\": float(values.get(\"foreground_mass\", 0.0)),\n        \"border\": float(values.get(\"border\", 0.0)),\n    }\n\n\n@torch.no_grad()\ndef sample_grid(generator, classifier, config: dict, device: torch.device, path: Path, n_per_class: int = 10) -> None:\n    \"\"\"Save a generated sample grid with classifier predictions.\"\"\"\n    generator.eval()\n    classifier.eval()\n    num_classes = int(config.get(\"num_classes\", 10))\n    z_dim = int(config.get(\"generator\", {}).get(\"z_dim\", 64))\n    images = []\n    labels = []\n    preds = []\n    for cls in range(num_classes):\n        y = torch.full((n_per_class,), cls, device=device, dtype=torch.long)\n        z = torch.randn(n_per_class, z_dim, device=device)\n        x = generator(z, y)\n        p = classifier(x).argmax(dim=1)\n        images.append(x.detach().cpu())\n        labels.extend([cls] * n_per_class)\n        preds.extend(p.detach().cpu().tolist())\n    plot_sample_grid_with_predictions(\n        torch.cat(images, dim=0),\n        torch.tensor(labels),\n        torch.tensor(preds),\n        n_per_class=n_per_class,\n        path=path,\n    )\n\n\n@torch.no_grad()\ndef fresh_eval_scalars(generator, classifier, feature_model, real_stats, config, device) -> dict[str, float]:\n    \"\"\"Cheap fresh-sample evaluation independent of training EMA.\"\"\"\n    generator.eval()\n    classifier.eval()\n    feature_model.eval()\n    eval_cfg = config.get(\"eval_during_train\", {})\n    num_classes = int(config.get(\"num_classes\", 10))\n    per_class = int(eval_cfg.get(\"per_class\", 64))\n    rounds = int(eval_cfg.get(\"rounds\", 2))\n    z_dim = int(config.get(\"generator\", {}).get(\"z_dim\", 64))\n    layer_weights = get_layer_weights(config)\n    logs = {}\n    ce_total = 0.0\n    correct = 0\n    total = 0\n    fd_layer_total = {layer: 0.0 for layer in real_stats}\n\n    for _ in range(rounds):\n        y = torch.arange(num_classes, device=device).repeat_interleave(per_class)\n        z = torch.randn(y.numel(), z_dim, device=device)\n        x = generator(z, y)\n        logits = classifier(x)\n        features = feature_model(x)\n        ce_total += F.cross_entropy(logits, y, reduction=\"sum\").item()\n        correct += (logits.argmax(dim=1) == y).sum().item()\n        total += y.numel()\n\n        from src.fd_loss import batch_moments_by_class\n\n        for layer, feats in features.items():\n            mu_b, m2_b = batch_moments_by_class(feats, y, num_classes)\n            layer_fd = 0.0\n            for cls in range(num_classes):\n                cov = cov_from_raw_moments(mu_b[cls], m2_b[cls], eps=float(config[\"train\"].get(\"covariance_epsilon\", 1e-4)))\n                terms = frechet_to_real(mu_b[cls], cov, real_stats[layer], cls)\n                layer_fd += float(terms[\"total\"].detach().cpu())\n            fd_layer_total[layer] += layer_fd / num_classes\n\n    logs[\"eval_ce_loss\"] = ce_total / total\n    logs[\"eval_classifier_fake_acc\"] = correct / total\n    logs[\"eval_fd_total_raw\"] = 0.0\n    for layer, value in fd_layer_total.items():\n        value = value / rounds\n        logs[f\"eval_fd_raw_{layer}\"] = value\n        logs[\"eval_fd_total_raw\"] += layer_weights.get(layer, 1.0) * value\n    return logs\n\n\ndef save_generator_checkpoint(generator, path: Path, config: dict, step: int) -> None:\n    \"\"\"Save a generator checkpoint.\"\"\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save({\"model_state_dict\": generator.state_dict(), \"config\": config, \"step\": step}, path)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--config\", default=ROOT / \"configs\" / \"generator_fd_mnist.yaml\")\n    parser.add_argument(\"--run_name\", default=\"layer_fd\")\n    args = parser.parse_args()\n\n    config = load_config(args.config)\n    set_seed(config.get(\"seed\"))\n    device = get_device(config.get(\"device\", \"auto\"))\n    run_dir = make_run_dir(config.get(\"run_root\", \"runs\"), args.run_name, root=ROOT)\n    save_config(config, run_dir / \"config.yaml\")\n    (run_dir / \"notes.md\").write_text(\n        \"Layer-aware FD-loss run.\\n\\n\"\n        \"Pipeline: train classifier -> compute real stats -> train generator -> evaluate generator.\\n\",\n        encoding=\"utf-8\",\n    )\n\n    data = load_data(config, root=ROOT)\n    classifier = build_classifier(config).to(device)\n    classifier_path = resolve_path(config[\"feature_model\"][\"checkpoint_path\"], ROOT)\n    load_state_dict(classifier, classifier_path, device)\n    classifier.eval()\n    for p in classifier.parameters():\n        p.requires_grad_(False)\n\n    feature_model = build_feature_extractor(classifier, config).to(device)\n    feature_model.eval()\n    for p in feature_model.parameters():\n        p.requires_grad_(False)\n\n    real_stats_path = resolve_path(config[\"real_stats_path\"], ROOT)\n    real_stats_obj = load_stats(real_stats_path, device=device)\n    real_stats = real_stats_obj[\"stats\"]\n    layer_names = list(real_stats.keys())\n    layer_dims = {layer: int(real_stats[layer][\"mu\"].shape[1]) for layer in layer_names}\n    layer_weights = get_layer_weights(config)\n\n    copy_if_exists(classifier_path, run_dir / \"classifier.pt\")\n    copy_if_exists(real_stats_path, run_dir / \"real_stats.pt\")\n    save_json({\"classifier_path\": str(classifier_path), \"real_stats_path\": str(real_stats_path)}, run_dir / \"artifact_refs.json\")\n\n    generator = build_generator(config).to(device)\n    gen_ckpt = resolve_path(config.get(\"generator\", {}).get(\"checkpoint_path\"), ROOT)\n    if gen_ckpt and gen_ckpt.exists():\n        load_state_dict(generator, gen_ckpt, device)\n\n    train_cfg = config[\"train\"]\n    z_dim = int(config[\"generator\"].get(\"z_dim\", 64))\n    ema_cfg = config.get(\"ema\", {})\n    ema_stats = init_generated_ema(\n        generator,\n        feature_model,\n        layer_names,\n        layer_dims,\n        num_classes=int(config.get(\"num_classes\", 10)),\n        z_dim=z_dim,\n        device=device,\n        rounds=int(ema_cfg.get(\"init_rounds\", 32)),\n        per_class=int(ema_cfg.get(\"init_per_class\", 64)),\n    )\n\n    opt = torch.optim.AdamW(\n        generator.parameters(),\n        lr=float(train_cfg.get(\"lr\", 1e-3)),\n        weight_decay=float(train_cfg.get(\"weight_decay\", 0.0)),\n    )\n\n    steps = int(train_cfg.get(\"steps\", 3000))\n    per_class = int(train_cfg.get(\"per_class\", 64))\n    num_classes = int(config.get(\"num_classes\", 10))\n    metrics = []\n    print(f\"run_dir={run_dir}\")\n\n    for step in range(1, steps + 1):\n        generator.train()\n        progress = step / steps\n        beta = lerp(float(ema_cfg.get(\"beta_start\", 0.9)), float(ema_cfg.get(\"beta_end\", 0.97)), progress)\n        y = torch.arange(num_classes, device=device).repeat_interleave(per_class)\n        z = torch.randn(y.numel(), z_dim, device=device)\n        x_fake = generator(z, y)\n        logits = classifier(x_fake)\n        features = feature_model(x_fake)\n\n        fd_loss_norm, fd_logs, new_ema = layer_weighted_fd_from_ema_batch(\n            features,\n            y,\n            ema_stats,\n            real_stats,\n            layer_weights,\n            beta=beta,\n            eps=float(train_cfg.get(\"covariance_epsilon\", 1e-4)),\n            num_classes=num_classes,\n            normalize=bool(train_cfg.get(\"normalize_fd\", True)),\n        )\n        ce_loss = F.cross_entropy(logits, y)\n        priors = config.get(\"image_priors\", {})\n        tv_loss = total_variation_loss(x_fake)\n        mass_loss = foreground_mass_loss(x_fake, target_mean=float(priors.get(\"foreground_target_mean\", 0.13)))\n        edge_loss = border_loss(x_fake, border=int(priors.get(\"border\", 3)))\n        weights = current_loss_weights(step, config)\n        total_loss = (\n            weights[\"fd\"] * fd_loss_norm\n            + weights[\"ce\"] * ce_loss\n            + weights[\"tv\"] * tv_loss\n            + weights[\"foreground_mass\"] * mass_loss\n            + weights[\"border\"] * edge_loss\n        )\n\n        opt.zero_grad(set_to_none=True)\n        total_loss.backward()\n        grad_norm = torch.nn.utils.clip_grad_norm_(generator.parameters(), float(train_cfg.get(\"grad_clip\", 5.0)))\n        opt.step()\n        ema_stats = new_ema\n\n        row = {\n            \"step\": step,\n            \"epoch_equiv\": step * y.numel() / len(data.train_dataset),\n            \"total_loss\": float(total_loss.detach().cpu()),\n            \"ce_loss\": float(ce_loss.detach().cpu()),\n            \"classifier_fake_acc\": float((logits.argmax(dim=1) == y).float().mean().detach().cpu()),\n            \"tv_loss\": float(tv_loss.detach().cpu()),\n            \"foreground_mass_loss\": float(mass_loss.detach().cpu()),\n            \"border_loss\": float(edge_loss.detach().cpu()),\n            \"x_mean\": float(x_fake.mean().detach().cpu()),\n            \"x_std\": float(x_fake.std().detach().cpu()),\n            \"grad_norm\": float(grad_norm),\n            \"beta\": beta,\n            \"lr\": get_lr(opt),\n            \"lambda_fd\": weights[\"fd\"],\n            \"lambda_ce\": weights[\"ce\"],\n            \"lambda_tv\": weights[\"tv\"],\n            \"lambda_foreground_mass\": weights[\"foreground_mass\"],\n            \"lambda_border\": weights[\"border\"],\n            **fd_logs,\n        }\n\n        if step == 1 or step % int(train_cfg.get(\"eval_every\", 250)) == 0:\n            row.update(fresh_eval_scalars(generator, classifier, feature_model, real_stats, config, device))\n\n        metrics.append(row)\n\n        if step == 1 or step % int(train_cfg.get(\"print_every\", 100)) == 0:\n            layer_log = \" \".join(f\"{layer}={row.get(f'fd_raw_{layer}', 0):.2f}\" for layer in layer_names)\n            print(\n                f\"step={step:05d} loss={row['total_loss']:.4f} \"\n                f\"fd_raw={row['fd_total_raw']:.2f} fd_norm={row['fd_total_norm']:.4f} \"\n                f\"{layer_log} ce={row['ce_loss']:.4f} acc={row['classifier_fake_acc']:.3f} \"\n                f\"beta={beta:.3f} grad={row['grad_norm']:.3f}\"\n            )\n\n        if step == 1 or step % int(train_cfg.get(\"sample_every\", 500)) == 0:\n            sample_grid(generator, classifier, config, device, run_dir / \"samples\" / f\"samples_step_{step:06d}.png\")\n\n        if step % int(train_cfg.get(\"checkpoint_every\", 1000)) == 0:\n            save_generator_checkpoint(generator, run_dir / f\"generator_step_{step:06d}.pt\", config, step)\n\n        if step == 1 or step % int(train_cfg.get(\"save_every\", 500)) == 0:\n            save_metrics_csv(metrics, run_dir / \"metrics.csv\")\n            plot_metrics_curves(run_dir / \"metrics.csv\", run_dir / \"plots\" / \"metrics_curves.png\")\n\n    save_generator_checkpoint(generator, run_dir / \"generator_final.pt\", config, steps)\n    save_metrics_csv(metrics, run_dir / \"metrics.csv\")\n    plot_metrics_curves(run_dir / \"metrics.csv\", run_dir / \"plots\" / \"metrics_curves.png\")\n    print(f\"saved final generator to {run_dir / 'generator_final.pt'}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/unpack_kaggle_run.py": "from __future__ import annotations\n\nimport argparse\nimport shutil\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef infer_run_name(zip_path: Path) -> str:\n    \"\"\"Infer the top-level run folder name inside a Kaggle zip.\"\"\"\n    with zipfile.ZipFile(zip_path) as zf:\n        top_levels = {\n            Path(name).parts[0]\n            for name in zf.namelist()\n            if name and not name.endswith(\"/\")\n        }\n    if len(top_levels) != 1:\n        raise ValueError(f\"Expected one top-level run folder in {zip_path}, found {sorted(top_levels)}\")\n    return next(iter(top_levels))\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"zip_path\", help=\"Downloaded fd_loss_mnist_run.zip\")\n    parser.add_argument(\"--runs_dir\", default=ROOT / \"runs\", help=\"Local runs directory\")\n    parser.add_argument(\"--overwrite\", action=\"store_true\", help=\"Replace an existing run folder\")\n    args = parser.parse_args()\n\n    zip_path = Path(args.zip_path).expanduser().resolve()\n    if not zip_path.exists():\n        raise FileNotFoundError(zip_path)\n\n    runs_dir = Path(args.runs_dir).expanduser().resolve()\n    runs_dir.mkdir(parents=True, exist_ok=True)\n\n    run_name = infer_run_name(zip_path)\n    dest = runs_dir / run_name\n    if dest.exists():\n        if not args.overwrite:\n            print(f\"Run already exists: {dest}\")\n            print(\"Pass --overwrite to replace it.\")\n            sys.exit(1)\n        shutil.rmtree(dest)\n\n    with zipfile.ZipFile(zip_path) as zf:\n        zf.extractall(runs_dir)\n\n    print(f\"Extracted run to: {dest}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "requirements.txt": "torch\ntorchvision\nnumpy\nscipy\npandas\nmatplotlib\npyyaml\n",
  "README.md": "# Layer-Aware FD-Loss MNIST\n\nThis is a small, notebook-friendly experiment codebase for layer-aware Frechet-distance-style generator training.\n\nThe pipeline is:\n\n```text\ntrain classifier -> compute real stats -> train generator -> evaluate generator -> analyze covariance/correlation/eigenspectra\n```\n\nThe current MNIST feature model exposes:\n\n- `early`: channel mean/std after the first conv block.\n- `middle`: channel mean/std after the second conv block.\n- `late`: 128-d penultimate dense feature.\n\nThe FD code does not assume these exact names. It consumes `dict[str, Tensor]` features from `src/features.py` and layer weights from config.\n\n## Commands\n\nRun from this directory:\n\n```bash\npython scripts/train_classifier.py --config configs/classifier_mnist.yaml\npython scripts/compute_real_stats.py --config configs/classifier_mnist.yaml\npython scripts/train_generator_fd.py --config configs/generator_fd_mnist.yaml --run_name layer_fd_test\npython scripts/evaluate_generator.py --run runs/<timestamp>_layer_fd_test\n```\n\n## Kaggle Workflow\n\nThe reliable Kaggle entrypoint is:\n\n```text\nnotebooks/kaggle_train_fd_loss_mnist.ipynb\n```\n\nIt is self-contained: when uploaded alone to Kaggle, it writes the needed `src/`, `scripts/`, and config files into `/kaggle/working/fd-loss-mnist`, runs the full experiment, and saves one downloadable artifact:\n\n```text\n/kaggle/working/fd_loss_mnist_run.zip\n```\n\nWorkflow:\n\n1. Upload `notebooks/kaggle_train_fd_loss_mnist.ipynb` to Kaggle.\n2. Run all cells.\n3. Download `/kaggle/working/fd_loss_mnist_run.zip` from the Output panel.\n4. Put it in this project folder or in Downloads.\n5. Unpack locally:\n\n```bash\npython scripts/unpack_kaggle_run.py fd_loss_mnist_run.zip\n```\n\nor:\n\n```bash\npython scripts/unpack_kaggle_run.py ~/Downloads/fd_loss_mnist_run.zip\n```\n\n6. Open a local analysis notebook and set:\n\n```python\nRUN = Path(\"../runs/<extracted_run_name>\")\n```\n\nThe zip contains the full run folder: config, classifier checkpoint, real stats, generator checkpoint, metrics, evaluation CSV/JSON, sample grids, plots, and notes.\n\n## Artifacts\n\nTraining runs create:\n\n```text\nruns/<timestamp>_<run_name>/\n  config.yaml\n  classifier.pt\n  real_stats.pt\n  generator_final.pt\n  metrics.csv\n  samples/\n  plots/\n  notes.md\n```\n\nEvaluation adds:\n\n```text\neval/\n  eval_summary.json\n  fd_by_layer_class.csv\n  generated_stats.pt\n  samples/\n  plots/\n```\n\nThe run folder should be enough to answer:\n\n> Which generator checkpoint, feature model checkpoint, real stats file, config, and seed produced this FD score?\n\n## Interpretation Notes\n\nEarly and middle MNIST features may show clearer covariance blocks because they are constructed from channel mean/std features. Late dense features have arbitrary index ordering, so raw covariance may look visually messier even when the representation is useful.\n\nFor structure, correlation plus hierarchical clustering is often more interpretable than raw covariance. When comparing real and generated matrices, the generated matrix must use the order computed from the real matrix; clustering each matrix separately can make generated structure look artificially aligned.\n\nEigenvalue spectra are often more informative for rank, diversity, and intrinsic dimensionality than trying to read block structure from dense late-layer covariance.\n"
}

for rel_path, content in BOOTSTRAP_FILES.items():
    path = PROJECT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

(PROJECT_DIR / "configs").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "runs").mkdir(parents=True, exist_ok=True)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Bootstrapped {len(BOOTSTRAP_FILES)} project files into {PROJECT_DIR}")

Bootstrapped 16 project files into /kaggle/working/fd-loss-mnist


In [3]:
# Kaggle-local configs. These are JSON on purpose, so the notebook does not depend on YAML parsing.
TRAIN_STEPS = int(os.environ.get("FD_TRAIN_STEPS", "3000"))
CLASSIFIER_EPOCHS = int(os.environ.get("FD_CLASSIFIER_EPOCHS", "10"))
if FAST_DEV_RUN:
    TRAIN_STEPS = 20
    CLASSIFIER_EPOCHS = 1

common_dataset = {
    "name": "mnist",
    "idx_path": "/kaggle/input",
    "torchvision_dir": str(WORKING_DIR / "data"),
    "train_batch_size": 256,
    "test_batch_size": 512,
    "num_workers": 0,
    "download": True,
}
feature_layers = [
    {"name": "early", "source": "pool1", "reducer": "channel_mean_std", "weight": 0.35},
    {"name": "middle", "source": "pool2", "reducer": "channel_mean_std", "weight": 0.65},
    {"name": "late", "source": "late", "reducer": "identity", "weight": 1.0},
]
classifier_dir = RUNS_DIR / "classifier_mnist"
classifier_config = {
    "seed": SEED,
    "device": "auto",
    "num_classes": 10,
    "dataset": common_dataset,
    "classifier": {
        "type": "mnist_cnn",
        "epochs": CLASSIFIER_EPOCHS,
        "lr": 0.001,
        "weight_decay": 0.0,
        "checkpoint_path": str(classifier_dir / "classifier.pt"),
        "history_path": str(classifier_dir / "classifier_history.csv"),
        "curves_path": str(classifier_dir / "classifier_training_curves.png"),
        "augment": True,
    },
    "features": {"type": "mnist_layer_features", "selected_layers": feature_layers},
    "stats": {"epsilon": 0.0001, "output_path": str(classifier_dir / "real_stats.pt")},
}

generator_config = {
    "seed": SEED,
    "device": "auto",
    "num_classes": 10,
    "run_root": str(RUNS_DIR),
    "dataset": common_dataset,
    "feature_model": {"type": "mnist_cnn", "checkpoint_path": str(classifier_dir / "classifier.pt")},
    "features": {"type": "mnist_layer_features", "selected_layers": feature_layers},
    "real_stats_path": str(classifier_dir / "real_stats.pt"),
    "generator": {"type": "one_step_mnist", "z_dim": 64, "checkpoint_path": None},
    "train": {
        "steps": TRAIN_STEPS,
        "per_class": 64,
        "lr": 0.001,
        "weight_decay": 0.0,
        "print_every": 100 if not FAST_DEV_RUN else 5,
        "eval_every": 250 if not FAST_DEV_RUN else 10,
        "save_every": 500 if not FAST_DEV_RUN else 10,
        "sample_every": 500 if not FAST_DEV_RUN else 10,
        "checkpoint_every": 1000,
        "grad_clip": 5.0,
        "covariance_epsilon": 0.0001,
        "normalize_fd": True,
    },
    "ema": {
        "init_rounds": 32 if not FAST_DEV_RUN else 2,
        "init_per_class": 64 if not FAST_DEV_RUN else 8,
        "beta_start": 0.90,
        "beta_end": 0.97,
    },
    "loss_weights": {
        "ce_warmup_steps": min(500, TRAIN_STEPS),
        "warmup": {"fd": 0.25, "ce": 2.0, "tv": 0.10, "foreground_mass": 2.0, "border": 0.50},
        "main": {"fd": 1.0, "ce": 0.15, "tv": 0.15, "foreground_mass": 2.0, "border": 0.50},
    },
    "image_priors": {"foreground_target_mean": 0.13, "border": 3},
    "eval_during_train": {"rounds": 2 if not FAST_DEV_RUN else 1, "per_class": 64 if not FAST_DEV_RUN else 8},
}

eval_config = {
    "seed": SEED + 1,
    "device": "auto",
    "num_classes": 10,
    "samples": {"per_class": 256 if not FAST_DEV_RUN else 16, "batch_per_class": 64 if not FAST_DEV_RUN else 8, "z_dim": 64},
    "plots": {"layers": ["early", "middle", "late"], "digits": [0, 1, 2], "matrix_percentile": 99.0, "cluster_method": "average"},
    "outputs": {"eval_dir": "eval"},
}

config_paths = {
    "classifier": PROJECT_DIR / "configs" / "kaggle_classifier_mnist.json",
    "generator": PROJECT_DIR / "configs" / "kaggle_generator_fd_mnist.json",
    "eval": PROJECT_DIR / "configs" / "kaggle_eval_mnist.json",
}
config_paths["classifier"].write_text(json.dumps(classifier_config, indent=2), encoding="utf-8")
config_paths["generator"].write_text(json.dumps(generator_config, indent=2), encoding="utf-8")
config_paths["eval"].write_text(json.dumps(eval_config, indent=2), encoding="utf-8")

print("Config files:")
for name, path in config_paths.items():
    print(f"  {name}: {path}")
print("TRAIN_STEPS:", TRAIN_STEPS)
print("CLASSIFIER_EPOCHS:", CLASSIFIER_EPOCHS)

Config files:
  classifier: /kaggle/working/fd-loss-mnist/configs/kaggle_classifier_mnist.json
  generator: /kaggle/working/fd-loss-mnist/configs/kaggle_generator_fd_mnist.json
  eval: /kaggle/working/fd-loss-mnist/configs/kaggle_eval_mnist.json
TRAIN_STEPS: 3000
CLASSIFIER_EPOCHS: 10


In [4]:
def run_cmd(args, cwd=PROJECT_DIR):
    print("\n$", " ".join(str(a) for a in args))
    result = subprocess.run([str(a) for a in args], cwd=str(cwd), text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {args}")

run_cmd([sys.executable, "scripts/train_classifier.py", "--config", config_paths["classifier"]])
run_cmd([sys.executable, "scripts/compute_real_stats.py", "--config", config_paths["classifier"]])


$ /usr/bin/python3 scripts/train_classifier.py --config /kaggle/working/fd-loss-mnist/configs/kaggle_classifier_mnist.json
Loading MNIST through torchvision at /kaggle/working/data


100%|██████████| 9.91M/9.91M [00:00<00:00, 42.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.03MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.7MB/s]


epoch 001/010 | train_loss=0.4801 train_acc=0.8508 | eval_loss=0.0801 eval_acc=0.9763
epoch 002/010 | train_loss=0.1240 train_acc=0.9618 | eval_loss=0.0497 eval_acc=0.9822
epoch 003/010 | train_loss=0.0918 train_acc=0.9713 | eval_loss=0.0399 eval_acc=0.9860
epoch 004/010 | train_loss=0.0736 train_acc=0.9766 | eval_loss=0.0389 eval_acc=0.9871
epoch 005/010 | train_loss=0.0618 train_acc=0.9810 | eval_loss=0.0286 eval_acc=0.9901
epoch 006/010 | train_loss=0.0547 train_acc=0.9833 | eval_loss=0.0253 eval_acc=0.9908
epoch 007/010 | train_loss=0.0513 train_acc=0.9837 | eval_loss=0.0266 eval_acc=0.9908
epoch 008/010 | train_loss=0.0469 train_acc=0.9855 | eval_loss=0.0295 eval_acc=0.9901
epoch 009/010 | train_loss=0.0438 train_acc=0.9862 | eval_loss=0.0270 eval_acc=0.9921
epoch 010/010 | train_loss=0.0405 train_acc=0.9873 | eval_loss=0.0253 eval_acc=0.9916
saved classifier checkpoint to /kaggle/working/runs/classifier_mnist/classifier.pt

$ /usr/bin/python3 scripts/compute_real_stats.py --confi

In [5]:
run_cmd([sys.executable, "scripts/train_generator_fd.py", "--config", config_paths["generator"], "--run_name", RUN_NAME])
matching_runs = sorted(RUNS_DIR.glob(f"*_{RUN_NAME}"))
if not matching_runs:
    raise FileNotFoundError(f"Could not find run matching *_{RUN_NAME} in {RUNS_DIR}")
RUN_DIR = matching_runs[-1]
shutil.copy2(config_paths["classifier"], RUN_DIR / "classifier_config.json")
shutil.copy2(config_paths["generator"], RUN_DIR / "generator_config.json")
print("RUN_DIR:", RUN_DIR)


$ /usr/bin/python3 scripts/train_generator_fd.py --config /kaggle/working/fd-loss-mnist/configs/kaggle_generator_fd_mnist.json --run_name layer_fd_kaggle
Loading MNIST through torchvision at /kaggle/working/data
run_dir=/kaggle/working/runs/20260508-205132_layer_fd_kaggle
step=00001 loss=16.5697 fd_raw=1510.58 fd_norm=2.0000 early=4.07 middle=27.15 late=1491.51 ce=7.3973 acc=0.100 beta=0.900 grad=1.274
step=00100 loss=0.6189 fd_raw=903.39 fd_norm=2.0000 early=0.52 middle=9.25 late=897.20 ce=0.0055 acc=1.000 beta=0.902 grad=0.166
step=00200 loss=0.5572 fd_raw=760.03 fd_norm=2.0000 early=0.16 middle=4.62 late=756.97 ce=0.0027 acc=1.000 beta=0.905 grad=0.043
step=00300 loss=0.5395 fd_raw=675.80 fd_norm=2.0000 early=0.13 middle=3.97 late=673.17 ce=0.0009 acc=1.000 beta=0.907 grad=0.035
step=00400 loss=0.5360 fd_raw=632.69 fd_norm=2.0000 early=0.12 middle=3.88 late=630.12 ce=0.0007 acc=1.000 beta=0.909 grad=0.030
step=00500 loss=0.5272 fd_raw=608.75 fd_norm=2.0000 early=0.10 middle=3.51 la

In [6]:
run_cmd([sys.executable, "scripts/evaluate_generator.py", "--run", RUN_DIR, "--config", config_paths["eval"]])
print("Evaluation outputs written under:", RUN_DIR / "eval")


$ /usr/bin/python3 scripts/evaluate_generator.py --run /kaggle/working/runs/20260508-205132_layer_fd_kaggle --config /kaggle/working/fd-loss-mnist/configs/kaggle_eval_mnist.json
evaluation summary:
  fd_total_weighted: 179.82778596681078
  num_classes: 10
  per_class: 256
  layer_names: ['early', 'middle', 'late']
  classifier_fake_acc: 0.996875
Evaluation outputs written under: /kaggle/working/runs/20260508-205132_layer_fd_kaggle/eval


In [7]:
# Ensure acceptance-critical files are present at the run root.
required = [
    RUN_DIR / "config.yaml",
    RUN_DIR / "classifier.pt",
    RUN_DIR / "classifier_config.json",
    RUN_DIR / "real_stats.pt",
    RUN_DIR / "generator_final.pt",
    RUN_DIR / "metrics.csv",
    RUN_DIR / "eval" / "eval_summary.json",
    RUN_DIR / "eval" / "fd_by_layer_class.csv",
    RUN_DIR / "samples" / "samples_final.png",
]

# The training script saves step sample grids; copy the newest one to samples_final.png.
final_sample = RUN_DIR / "samples" / "samples_final.png"
if not final_sample.exists():
    sample_candidates = sorted((RUN_DIR / "samples").glob("samples_step_*.png"))
    if sample_candidates:
        shutil.copy2(sample_candidates[-1], final_sample)

# Convenience root copies for the requested artifact contract.
for name in ["eval_summary.json", "fd_by_layer_class.csv"]:
    src = RUN_DIR / "eval" / name
    dst = RUN_DIR / name
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))

zip_base = Path("/kaggle/working/fd_loss_mnist_run") if IS_KAGGLE else WORKING_DIR / "fd_loss_mnist_run"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
print("Saved artifact:", zip_path)
print("\nImportant files:")
for rel in [
    "classifier.pt",
    "classifier_config.json",
    "real_stats.pt",
    "generator_final.pt",
    "metrics.csv",
    "eval_summary.json",
    "fd_by_layer_class.csv",
    "samples/samples_final.png",
    "plots/metrics_curves.png",
    "plots/generated_confusion_matrix.png",
]:
    path = RUN_DIR / rel
    print(f"  {rel}: {'OK' if path.exists() else 'missing'}")

Saved artifact: /kaggle/working/fd_loss_mnist_run.zip

Important files:
  classifier.pt: OK
  classifier_config.json: OK
  real_stats.pt: OK
  generator_final.pt: OK
  metrics.csv: OK
  eval_summary.json: OK
  fd_by_layer_class.csv: OK
  samples/samples_final.png: OK
  plots/metrics_curves.png: OK
  plots/generated_confusion_matrix.png: OK
